# Clase 156 — Autoencoders: undercomplete, stacked, denoising, sparse

Un **autoencoder** aprende `Encoder → bottleneck → Decoder` reconstruyendo su
propio input. Variantes: **undercomplete** (`latent < input`), **stacked**
(profundo), **denoising** (input ruidoso → output limpio) y **sparse**
(penaliza activaciones latentes con L1).

**Requiere:** `tensorflow` / `keras`. Si no está instalado, las celdas avisan; el
código es la **API real de Keras** (no se ejecuta sin TF).

## 1. Entorno + datos

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, regularizers
    HAS_TF = True
    tf.random.set_seed(42)
    print('tensorflow:', tf.__version__)
except Exception as e:
    HAS_TF = False
    print('tensorflow no instalado. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)

## 2. Datos: Fashion-MNIST (o sintético)

Se normalizan las imágenes a `[0, 1]` y se aplanan a 784.

In [ ]:
if HAS_TF:
    (X_train, _), (X_test, _) = keras.datasets.fashion_mnist.load_data()
    X_train = X_train.astype('float32') / 255.0
    X_test = X_test.astype('float32') / 255.0
    X_train = X_train.reshape(-1, 784)
    X_test = X_test.reshape(-1, 784)
else:
    X_train = np.random.rand(1000, 784).astype('float32')
    X_test = np.random.rand(200, 784).astype('float32')
print('train:', X_train.shape)

## 3. Undercomplete AE (`784 → 64 → 784`)

El bottleneck de 64 (< 784) fuerza compresión. Se entrena con MSE.

In [ ]:
if HAS_TF:
    encoder = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu', name='bottleneck'),
    ])
    decoder = keras.Sequential([
        keras.Input(shape=(64,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(784, activation='sigmoid'),
    ])
    ae = keras.Sequential([encoder, decoder])
    ae.compile(optimizer='adam', loss='mse')
    ae.summary()
    # ae.fit(X_train, X_train, epochs=10, batch_size=256, validation_split=0.1)
else:
    print('Encoder 784->128->64 ; Decoder 64->128->784(sigmoid) ; loss=MSE')

## 4. Stacked / deep AE con más capas

Más capas Dense = mayor capacidad de compresión no lineal (PCA no lineal).

In [ ]:
if HAS_TF:
    stacked = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu', name='code'),   # codigo mas chico
        layers.Dense(64, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(784, activation='sigmoid'),
    ])
    stacked.compile(optimizer='adam', loss='binary_crossentropy')
    print('capas:', len(stacked.layers))
else:
    print('Stacked AE simetrico: 784-256-64-32-64-256-784')

## 5. Denoising AE: input `x + ruido`, target `x`

Aprende invariancias reconstruyendo la señal limpia desde una versión corrupta.

In [ ]:
noise_factor = 0.5
X_noisy = X_train + noise_factor * np.random.normal(size=X_train.shape).astype('float32')
X_noisy = np.clip(X_noisy, 0.0, 1.0)

if HAS_TF:
    denoiser = keras.models.clone_model(ae)
    denoiser.compile(optimizer='adam', loss='mse')
    # se entrena input ruidoso -> target limpio
    # denoiser.fit(X_noisy, X_train, epochs=10, batch_size=256)
    print('Denoising AE: fit(X_noisy, X_train) — target es la imagen limpia.')
else:
    print('input = x + 0.5*ruido (clip 0..1) ; target = x')

## 6. Sparse AE: L1 sobre las activaciones latentes

`activity_regularizer=regularizers.l1(1e-3)` penaliza `‖latent‖₁` → pocas
neuronas activas por muestra.

In [ ]:
if HAS_TF:
    sparse = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu',
                     activity_regularizer=regularizers.l1(1e-3), name='sparse_code'),
        layers.Dense(128, activation='relu'),
        layers.Dense(784, activation='sigmoid'),
    ])
    sparse.compile(optimizer='adam', loss='mse')
    print('Sparse AE con activity_regularizer=l1(1e-3) en el bottleneck.')
else:
    print('regularizers.l1(1e-3) como activity_regularizer del codigo latente.')

## 7. Anomaly detection con el error de reconstrucción

Alta reconstruction error ⇒ la muestra no se parece a lo visto en entrenamiento.

In [ ]:
def reconstruction_error(model, X):
    if HAS_TF:
        recon = model.predict(X, verbose=0)
    else:
        recon = X + np.random.normal(0, 0.1, X.shape)   # placeholder sin TF
    return np.mean((X - recon) ** 2, axis=1)

# score = MSE por muestra; umbral alto -> anomalia
err = reconstruction_error(ae if HAS_TF else None, X_test[:50])
print('reconstruction error (5 primeras):', np.round(err[:5], 4))

## Ejercicios

1. **AE simple**: entrená `784→64→784` en MNIST y visualizá reconstrucciones.
2. **Latent 2D**: con `latent_dim=2`, graficá scatter de 1000 imágenes coloreado por clase.
3. **Denoising**: agregá `0.5·ruido` y mostrá que reconstruye limpio.
4. **Sparse**: agregá `l1(1e-3)` al latente e inspeccioná las activaciones.
5. **Anomaly detection**: entrená solo sobre una clase y usá el MSE como score (ROC-AUC ≥ 0.85).

## Conclusiones

- Un AE comprime en el bottleneck y reconstruye; MSE/BCE como reconstruction loss.
- Undercomplete (`latent < input`) fuerza compresión; sparse la logra con L1
  aunque el latente sea grande.
- El denoising AE aprende invariancias reconstruyendo desde inputs corruptos.
- El reconstruction error es un score directo de **anomaly detection**.
- El latent space de un AE no es continuo → mal generador, lo que motiva el VAE.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

### Ejercicio 1 — AE simple `784→64→784` (API Keras) + AE lineal ejecutable

In [ ]:
# API real Keras (undercomplete AE, entrenar en MNIST):
if HAS_TF:
    ae.fit(X_train, X_train, epochs=10, batch_size=256, validation_split=0.1)
    recon = ae.predict(X_test[:8], verbose=0)
    print('reconstrucciones:', recon.shape)
else:
    print('ae.fit(X_train, X_train, epochs=10) ; ae.predict -> reconstrucciones.')

# Núcleo ejecutable: un AE LINEAL optimo = PCA (via SVD). Mas dimensiones de
# codigo => menor error de reconstruccion.
import numpy as np
rng = np.random.default_rng(0)
Z = rng.normal(size=(500, 3)); Wtrue = rng.normal(size=(3, 20))
Xd = Z @ Wtrue; Xd = Xd - Xd.mean(0)              # datos de rango 3 embebidos en 20D
def linear_ae_recon(X, k):
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    Xk = (U[:, :k] * S[:k]) @ Vt[:k]              # proyeccion a k componentes
    return float(((X - Xk) ** 2).mean())
e2, e3 = linear_ae_recon(Xd, 2), linear_ae_recon(Xd, 3)
print(f'error recon  k=2: {e2:.4f}   k=3: {e3:.2e}')
assert e3 < e2 and e3 < 1e-6      # k=3 = rango real -> reconstruccion perfecta

### Ejercicio 2 — Latent space 2D coloreado por clase

In [ ]:
if HAS_TF:
    from tensorflow import keras
    enc2 = keras.Sequential([keras.Input(shape=(784,)),
                             keras.layers.Dense(64, activation='relu'),
                             keras.layers.Dense(2, name='latent2d')])
    dec2 = keras.Sequential([keras.Input(shape=(2,)),
                             keras.layers.Dense(64, activation='relu'),
                             keras.layers.Dense(784, activation='sigmoid')])
    ae2 = keras.Sequential([enc2, dec2]); ae2.compile('adam', 'mse')
    # ae2.fit(X_train, X_train, epochs=10)
    # Z = enc2.predict(X_train[:1000]); plt.scatter(Z[:,0], Z[:,1], c=labels)
    print('latent_dim=2 -> scatter de 1000 imagenes coloreado por clase.')
else:
    print('Encoder a 2D -> scatter del codigo; clases se agrupan por regiones.')

### Ejercicio 3 — Denoising AE: input ruidoso → target limpio

In [ ]:
if HAS_TF:
    denoiser.fit(X_noisy, X_train, epochs=10, batch_size=256)   # target = limpio
    clean = denoiser.predict(X_noisy[:8], verbose=0)
    print('denoised:', clean.shape)
else:
    import numpy as np
    # Verificacion del setup: el input ruidoso difiere del limpio, el target es limpio.
    diff = float(np.mean((X_noisy - X_train) ** 2))
    print(f'MSE(input_ruidoso, limpio) = {diff:.4f} (>0)')
    assert diff > 0
    print('El AE aprende f(x+ruido) -> x reconstruyendo la señal limpia.')

### Ejercicio 4 — Sparse AE: L1 sobre el código latente

In [ ]:
if HAS_TF:
    sparse.fit(X_train, X_train, epochs=10, batch_size=256)
    from tensorflow import keras
    code_model = keras.Model(sparse.input, sparse.get_layer('sparse_code').output)
    acts = code_model.predict(X_test[:100], verbose=0)
    print('fraccion de activaciones ~0:', float((acts < 1e-3).mean()))
else:
    print('activity_regularizer=l1(1e-3) -> pocas neuronas activas por muestra'
          ' (codigo disperso), aunque el bottleneck sea grande.')

### Ejercicio 5 — Anomaly detection: reconstruction error como score (ejecutable)

In [ ]:
# Núcleo ejecutable: AE lineal entrenado en clase 'normal'; el MSE separa anomalias.
import numpy as np
rng = np.random.default_rng(1)
normal = rng.normal(0, 1, size=(400, 10)) @ rng.normal(size=(10, 30))   # rango 10 -> 30D
anom = rng.normal(0, 1, size=(80, 30)) * 3                              # ruido full-rank
mu = normal.mean(0)
U, S, Vt = np.linalg.svd(normal - mu, full_matrices=False)
P = Vt[:10]                                            # subespacio 'normal'
def recon_err(X):
    Xc = X - mu
    return ((Xc - Xc @ P.T @ P) ** 2).mean(axis=1)
err_n, err_a = recon_err(normal), recon_err(anom)
print(f'error normal medio={err_n.mean():.3f}  anomalia medio={err_a.mean():.3f}')
# ROC-AUC via probabilidad de que una anomalia tenga mayor error que una normal
auc = float((err_a[:, None] > err_n[None, :]).mean())
print(f'ROC-AUC (error como score) = {auc:.2f}')
assert auc >= 0.85